# Inference on the Full Corpus

This notebook executes inference using the fine-tuned Named Entity Recognition (NER) model (`checkpoints/final/saved_model`) across all unannotated corpus sentences, extending entity coverage beyond the 305 gold-standard sentences.

**Prerequisite:** `model.ipynb` must be executed prior to running this notebook to generate the model checkpoint.

In [1]:
import os
import csv

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
if os.path.basename(NOTEBOOK_DIR) == "notebooks":
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
else:
    PROJECT_ROOT = NOTEBOOK_DIR

FINAL_ANNOTATION_PATH = os.path.join(PROJECT_ROOT, "data", "annotations", "final_annotation.csv")
SENTENCE_SCORES_PATH = os.path.join(PROJECT_ROOT, "data", "annotations", "sentence_scores.csv")
FINAL_MODEL_DIR = os.path.join(PROJECT_ROOT, "checkpoints", "final", "saved_model")

INFERRED_DIR = os.path.join(PROJECT_ROOT, "data", "model_predictions")
OUTPUT_PATH = os.path.join(INFERRED_DIR, "inferred_entities.csv")
os.makedirs(INFERRED_DIR, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.6  # same value validated in model.ipynb

ENTITY_TYPES = ["FLORA", "FAUNA", "WEATHER", "LANDSCAPE", "NATURE"]
label_list = ["O"] + [f"{p}-{t}" for t in ENTITY_TYPES for p in ("B", "I")]
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

final_model = AutoModelForTokenClassification.from_pretrained(FINAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)

## Identification of Unannotated Sentences

Sentence IDs from `final_annotation.csv` are cross-referenced against all parsed sentences in `sentence_scores.csv` to isolate the unannotated corpus subset.

In [2]:
def load_annotated_sentence_ids(path):
    with open(path, newline="", encoding="utf-8") as f:
        return {row["sentence_id"] for row in csv.DictReader(f)}


def load_unannotated_sentences(sentence_scores_path, annotated_ids):
    with open(sentence_scores_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [r for r in rows if r["sentence_id"] not in annotated_ids]


annotated_ids = load_annotated_sentence_ids(FINAL_ANNOTATION_PATH)
unannotated = load_unannotated_sentences(SENTENCE_SCORES_PATH, annotated_ids)

print(f"Annotated (gold-standard) sentence IDs: {len(annotated_ids)}")
print(f"Unannotated sentences to run inference on: {len(unannotated)}")

Annotated (gold-standard) sentence IDs: 305
Unannotated sentences to run inference on: 575


## Inference functions

*   `decode_bio_to_spans()` converts sequence-level BIO predictions back into original character-level span offsets.
*   `predict_with_threshold()` processes single sentences and applies the `CONFIDENCE_THRESHOLD` (0.6). Predictions below this threshold are mapped to the background `"O"` class, matching the post-processing pipeline established during model evaluation.

In [3]:
def decode_bio_to_spans(offsets, word_ids, labels, id_to_label):
    spans, current, current_word_id = [], None, None
    previous_word_id = None
    for (start, end), wid, lab_id in zip(offsets, word_ids, labels):
        if wid is None:
            previous_word_id = wid
            continue
        if wid == previous_word_id:
            if current is not None and wid == current_word_id:
                current["end"] = end
            previous_word_id = wid
            continue
        if lab_id != -100:
            lab = id_to_label[lab_id]
            if lab == "O":
                if current: spans.append(current); current = None
            elif lab.startswith("B-"):
                if current: spans.append(current)
                current = {"start": start, "end": end, "label": lab[2:]}
                current_word_id = wid
            elif lab.startswith("I-"):
                if current and current["label"] == lab[2:]:
                    current["end"] = end; current_word_id = wid
                else:
                    if current: spans.append(current)
                    current = {"start": start, "end": end, "label": lab[2:]}
                    current_word_id = wid
        previous_word_id = wid
    if current: spans.append(current)
    return spans


def predict_with_threshold(text, model, tokenizer, label_to_id, id_to_label, threshold=CONFIDENCE_THRESHOLD):
    device = next(model.parameters()).device
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True, return_tensors="pt")
    offsets = encoding["offset_mapping"][0].tolist()
    word_ids = encoding.word_ids(batch_index=0)
    model_inputs = {k: v.to(device) for k, v in encoding.items() if k != "offset_mapping"}

    model.eval()
    with torch.no_grad():
        logits = model(**model_inputs).logits[0]
    probs = torch.softmax(logits, dim=-1)
    confidences, pred_ids = torch.max(probs, dim=-1)

    o_id = label_to_id["O"]
    pred_ids = torch.where(confidences < threshold, torch.full_like(pred_ids, o_id), pred_ids)

    return decode_bio_to_spans(offsets, word_ids, pred_ids.tolist(), id_to_label)


# test
predict_with_threshold("The wind swept over the mountain.", final_model, tokenizer, label_to_id, id_to_label)

[{'start': 4, 'end': 8, 'label': 'WEATHER'},
 {'start': 24, 'end': 32, 'label': 'LANDSCAPE'}]

## Full-Corpus Inference Execution

Runs `predict_with_threshold()` over every sentence, returning one row per predicted entity.

In [4]:
def run_inference_on_corpus(unannotated_sentences, model, tokenizer, label_to_id, id_to_label):
    results = []
    for row in unannotated_sentences:
        sentence_id = row["sentence_id"]
        text = row["sentence_text"]
        spans = predict_with_threshold(text, model, tokenizer, label_to_id, id_to_label)

        if not spans:
            # no entities predicted - still record the sentence
            results.append({
                "sentence_id": sentence_id, "sentence_text": text,
                "target_span": "", "start_char": "", "end_char": "", "entity_type": "",
            })
        else:
            for s in spans:
                results.append({
                    "sentence_id": sentence_id, "sentence_text": text,
                    "target_span": text[s["start"]:s["end"]],
                    "start_char": s["start"], "end_char": s["end"],
                    "entity_type": s["label"],
                })
    return results


inferred = run_inference_on_corpus(unannotated, final_model, tokenizer, label_to_id, id_to_label)

n_sentences_with_entities = len({r["sentence_id"] for r in inferred if r["target_span"]})
n_total_entities = sum(1 for r in inferred if r["target_span"])
print(f"Ran inference on {len(unannotated)} sentences")
print(f"  {n_sentences_with_entities} received at least one predicted entity")
print(f"  {n_total_entities} entities predicted in total")

Ran inference on 575 sentences
  107 received at least one predicted entity
  164 entities predicted in total


## Entity Density Verification

Initial inference yields 164 entities across 575 sentences (0.29 entities/sentence), compared to 797 entities across 305 sentences (2.61 entities/sentence) in the gold-standard dataset. 

To evaluate whether this lower density reflects model under-prediction or true content sparsity, entity detection rates are cross-referenced against the WordNet candidate counts (`n_candidates` bands) from `sentence_scores.csv`.

**Key Findings:**
*   **Corpus Sparsity:** No high-density sentences (`band_2` or `band_3plus`) remain in the unannotated pool. Stratified sampling during dataset creation successfully selected nearly all dense sentences for manual annotation.
*   **Lexicon Generalization:** Sentences in `band_0` (zero initial WordNet candidates) produced entity predictions 12.8% of the time, demonstrating model capacity to extract novel entity spans beyond static lexicon lookup.
*   **Density Alignment:** The average WordNet candidate count drops from 2.13 in the annotated set to 0.13 in the unannotated set. Relative to this baseline, entities are extracted at a higher proportional rate than the initial lexicon heuristic predicted.

In [5]:
scores_by_id = {}
with open(SENTENCE_SCORES_PATH, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        scores_by_id[row["sentence_id"]] = row

entity_counts_by_id = {}
for r in inferred:
    if r["target_span"]:
        entity_counts_by_id[r["sentence_id"]] = entity_counts_by_id.get(r["sentence_id"], 0) + 1

from collections import defaultdict
band_totals = defaultdict(lambda: [0, 0])  # band -> [sentences, sentences_with_predictions]
for row in unannotated:
    sid = row["sentence_id"]
    band = scores_by_id[sid]["band"]
    band_totals[band][0] += 1
    if entity_counts_by_id.get(sid, 0) > 0:
        band_totals[band][1] += 1

print(f"{'band':12s} {'sentences':>10s} {'got predictions':>18s} {'%':>8s}")
for band in sorted(band_totals):
    total, with_pred = band_totals[band]
    print(f"{band:12s} {total:>10d} {with_pred:>18d} {100*with_pred/total:>7.1f}%")

# average annotated-vs-unannotated WordNet density
annotated_scores = [int(scores_by_id[sid]["n_candidates"]) for sid in annotated_ids if sid in scores_by_id]
unannotated_scores = [int(row["n_candidates"]) for row in unannotated]
print(f"\nAverage WordNet n_candidates - annotated set: {sum(annotated_scores)/len(annotated_scores):.2f}")
print(f"Average WordNet n_candidates - unannotated set: {sum(unannotated_scores)/len(unannotated_scores):.2f}")

band          sentences    got predictions        %
band_0              499                 64    12.8%
band_1               76                 43    56.6%

Average WordNet n_candidates - annotated set: 2.13
Average WordNet n_candidates - unannotated set: 0.13


## Exporting Results

Predictions are serialized to `inferred_entities.csv` -> same column structure as `final_annotation.csv`'s target columns

In [6]:
fieldnames = ["sentence_id", "sentence_text", "target_span", "start_char", "end_char", "entity_type"]

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(inferred)

print(f"Wrote {len(inferred)} rows to {OUTPUT_PATH}")
print(f"  ({n_sentences_with_entities} sentences with entities, "
      f"{len(unannotated) - n_sentences_with_entities} sentences with none)")

Wrote 632 rows to /Users/sara/Documents/GitHub/understory/data/model_predictions/inferred_entities.csv
  (107 sentences with entities, 468 sentences with none)
